In [1]:
import torch
import torch.nn as nn
def main():
    print("Hello from torch-test!")
    if torch.backends.mps.is_available():
        print("Excellent! MPS backend is available.")
    else:
        print("MPS backend is not available: Something went wrong! Are you running this on a Mac with Apple Silicon chip?")

if __name__ == "__main__":
    main()

## Normalization × Tensor
结合第 14 页的 LLM 示例与板书：着色竖条是 X[b, s, :]。LayerNorm 与 RMSNorm 都沿 hidden 维独立计算，但统计量不同。

![LLM 的归一化范围](../../images/2026/l5/llm_normalization.png)

固定 b、s，对 X[b,s,:] 的 H 个特征统计；keepdim=True 保留 [B,S,1]，广播后仍为 [B,S,H]。
gamma 的形状是 [H]，同一层所有 batch 和 token 共享它。


In [2]:
# 课堂小张量示例：2 个 batch × 3 个 token × 4 个特征
x = torch.tensor([[[1., 2., 1., 2.], [2., 4., 2., 4.], [0., -3., 0., 3.]],
                  [[0., 1., 0., 1.], [1., -1., 1., -1.], [3., 0., 4., 0.]]])
variance_epsilon = 1e-6
rms = torch.sqrt(x.square().mean(-1, keepdim=True) + variance_epsilon)
print("输入 shape：", tuple(x.shape))
print("每个 token 的 RMS（shape [2, 3, 1]）：\n", rms)
print("逐行缩放（gamma=1）：\n", x / rms)
print("缩放后的 RMS：\n", (x / rms).square().mean(-1).sqrt())


输入 shape： (2, 3, 4)
每个 token 的 RMS（shape [2, 3, 1]）：
 tensor([[[1.5811],
         [3.1623],
         [2.1213]],

        [[0.7071],
         [1.0000],
         [2.5000]]])
逐行缩放（gamma=1）：
 tensor([[[ 0.6325,  1.2649,  0.6325,  1.2649],
         [ 0.6325,  1.2649,  0.6325,  1.2649],
         [ 0.0000, -1.4142,  0.0000,  1.4142]],

        [[ 0.0000,  1.4142,  0.0000,  1.4142],
         [ 1.0000, -1.0000,  1.0000, -1.0000],
         [ 1.2000,  0.0000,  1.6000,  0.0000]]])
缩放后的 RMS：
 tensor([[1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000]])


## 对齐手写与内置 RMSNorm
统一 eps、gamma；高精度计算后恢复输入 dtype。缩放后的单位 RMS 结论只适用于 gamma=1（忽略 eps 的影响）。

In [3]:
# 对齐 eps 与可学习 gamma，比较手写版和 PyTorch
norm = nn.RMSNorm(4, eps=variance_epsilon)
with torch.no_grad():
    norm.weight.copy_(torch.tensor([1., 0.5, 1.5, 2.]))
x32 = x.float()
mean_square = x32.square().mean(-1, keepdim=True)
normalized = (x32 * torch.rsqrt(mean_square + norm.eps)).to(x.dtype)
y_manual = normalized * norm.weight
y_torch = norm(x)
torch.testing.assert_close(y_manual, y_torch)
print("gamma：", norm.weight.detach())
print("相同 eps、gamma 下的最大误差：", (y_manual-y_torch).abs().max().item())
print("非单位 gamma 会重新改变各维的尺度：\n", y_torch.detach())


gamma： tensor([1.0000, 0.5000, 1.5000, 2.0000])
相同 eps、gamma 下的最大误差： 0.0
非单位 gamma 会重新改变各维的尺度：
 tensor([[[ 0.6325,  0.6325,  0.9487,  2.5298],
         [ 0.6325,  0.6325,  0.9487,  2.5298],
         [ 0.0000, -0.7071,  0.0000,  2.8284]],

        [[ 0.0000,  0.7071,  0.0000,  2.8284],
         [ 1.0000, -0.5000,  1.5000, -2.0000],
         [ 1.2000,  0.0000,  2.4000,  0.0000]]])


## LayerNorm 与 RMSNorm 的选择理由
LayerNorm 调整中心和尺度；RMSNorm 省去中心化，计算更简单。原论文在所测任务中报告了相近的效果，实际速度收益依赖实现和硬件。
参考：[RMSNorm 原论文](https://arxiv.org/abs/1910.07467)。这是效果与计算成本的取舍，不意味着均值永远不重要或 LayerNorm 失效。


## FFN：升维、门控与降维
FFN 是 Transformer 的前馈子层，通常由 MLP 实现；Llama 中的代码类名为 LlamaMLP。本实验统一称 FFN。
对应讲义 FFN 原理页。把下面的 h 看作 attention 与残差之后，再经 RMSNorm 的输入。
权重用手工数值便于观察，不是预训练模型的语义解释。H=2、F=4。


In [4]:
import torch.nn.functional as F
up = nn.Linear(2, 4, bias=False)
gate = nn.Linear(2, 4, bias=False)
down = nn.Linear(4, 2, bias=False)
with torch.no_grad():
    up.weight.copy_(torch.tensor([[1.,0.], [0.,1.], [1.,1.], [1.,-1.]]))
    gate.weight.copy_(torch.tensor([[1.,0.], [-1.,0.], [0.,1.], [0.,-1.]]))
    down.weight.copy_(torch.tensor([[1.,0.,.5,0.], [0.,1.,0.,.5]]))
t = torch.tensor([[[3.,4.], [4.,3.]]])  # attention 与第一次残差后的主干表示
norm_ffn = nn.RMSNorm(2, eps=1e-6)
with torch.no_grad():
    h = norm_ffn(t)  # 交给 FFN 的输入；保留 t 以供残差相加
with torch.no_grad():
    u = up(h)
    g = F.silu(gate(h))
    z = g * u
    delta_ffn = down(z)
for name, value in [("h",h), ("候选特征 u",u), ("门控系数 g",g), ("调节后的 z",z), ("hidden 更新 delta_ffn",delta_ffn)]:
    print(name, "shape=", tuple(value.shape), "\n", value)
print("两个 token 使用同一组权重，但产生不同的门控系数。")


h shape= (1, 2, 2) 
 tensor([[[0.8485, 1.1314],
         [1.1314, 0.8485]]])
候选特征 u shape= (1, 2, 4) 
 tensor([[[ 0.8485,  1.1314,  1.9799, -0.2828],
         [ 1.1314,  0.8485,  1.9799,  0.2828]]])
门控系数 g shape= (1, 2, 4) 
 tensor([[[ 0.5942, -0.2543,  0.8554, -0.2760],
         [ 0.8554, -0.2760,  0.5942, -0.2543]]])
调节后的 z shape= (1, 2, 4) 
 tensor([[[ 0.5042, -0.2878,  1.6936,  0.0781],
         [ 0.9678, -0.2342,  1.1764, -0.0719]]])
hidden 更新 delta_ffn shape= (1, 2, 2) 
 tensor([[[ 1.3510, -0.2487],
         [ 1.5560, -0.2701]]])
两个 token 使用同一组权重，但产生不同的门控系数。


### 降维是加权组合；只有连续线性投影可以合并
先比较 down(up(h)) 与合并矩阵。加入 SiLU 和输入相关乘法后，完整 FFN 不再是这个线性映射。


In [5]:
with torch.no_grad():
    linear_only = down(up(h))
    merged = F.linear(h, down.weight @ up.weight)
    torch.testing.assert_close(linear_only, merged)
    expanded_sum = torch.einsum("bsf,hf->bsh", z, down.weight)
    torch.testing.assert_close(delta_ffn, expanded_sum)
print("纯线性升维/降维可合并；down 实际加权组合所有中间特征。")
print("纯线性结果：", linear_only)
print("门控 FFN 结果：", delta_ffn)


纯线性升维/降维可合并；down 实际加权组合所有中间特征。
纯线性结果： tensor([[[1.8385, 0.9899],
         [2.1213, 0.9899]]])
门控 FFN 结果： tensor([[[ 1.3510, -0.2487],
         [ 1.5560, -0.2701]]])


### FFN 逐 token 计算，更新通过残差写回
修改第二个 token，不应改变第一个 token 的 FFN 输出。上面已计算 h = Norm(t)；这里将更新 delta_ffn 加回保留的主干 t。


In [6]:
with torch.no_grad():
    changed = h.clone()
    changed[:,1,:] = torch.tensor([-3.,5.])
    changed_delta = down(F.silu(gate(changed)) * up(changed))
    torch.testing.assert_close(delta_ffn[:,0], changed_delta[:,0])
    y = t + delta_ffn
assert y.shape == h.shape
print("第二个 token 改变，第一个 token 的 FFN 输出保持不变。")
print("残差更新 t + delta_ffn：", y)


第二个 token 改变，第一个 token 的 FFN 输出保持不变。
残差更新 t + delta_ffn： tensor([[[4.3510, 3.7513],
         [5.5560, 2.7299]]])


## 最后补充：高维 RoPE——前后半维配对
对应讲义 rotate_half。相邻配对与前后半维配对是不同布局，Q/K 与 cos/sin 必须使用一致的布局。

In [7]:
# Llama 风格：前后半维配对，Q/K shape [batch, seq, heads, dim]
def build_rope_cache(seq_len, dim, base=10000):
    assert dim % 2 == 0
    inv_freq = base ** (-torch.arange(0, dim, 2, dtype=torch.float32) / dim)
    angles = torch.outer(torch.arange(seq_len, dtype=torch.float32), inv_freq)
    angles = torch.cat((angles, angles), dim=-1)
    return angles.cos(), angles.sin()

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    cos, sin = cos[None, :, None, :], sin[None, :, None, :]
    return q*cos + rotate_half(q)*sin, k*cos + rotate_half(k)*sin

q = torch.arange(64, dtype=torch.float32).reshape(1, 4, 2, 8)
k = q + 1
cos, sin = build_rope_cache(4, 8)
q_rot, k_rot = apply_rotary_pos_emb(q, k, cos, sin)
torch.testing.assert_close(q_rot.norm(dim=-1), q.norm(dim=-1))
torch.testing.assert_close(k_rot.norm(dim=-1), k.norm(dim=-1))
torch.testing.assert_close(q_rot[:, 0], q[:, 0])
print("Q 输入/输出 shape：", tuple(q.shape), tuple(q_rot.shape))
print("位置 1、第 0 个 head：", q[0,1,0], "→", q_rot[0,1,0])
print("长度保持；位置 0 不旋转。配对为 (x0,x4)、(x1,x5)、(x2,x6)、(x3,x7)。")


Q 输入/输出 shape： (1, 4, 2, 8) (1, 4, 2, 8)
位置 1、第 0 个 head： tensor([16., 17., 18., 19., 20., 21., 22., 23.]) → tensor([-8.1846, 14.8186, 17.7791, 18.9770, 24.2696, 22.5923, 22.1789, 23.0190])
长度保持；位置 0 不旋转。配对为 (x0,x4)、(x1,x5)、(x2,x6)、(x3,x7)。
